# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Classification — predicting content decline.**

FlyRank manages thousands of pages across 32 clients. Content that once ranked well in Google Search quietly decays — rankings slip, clicks drop, and most teams notice too late. The most valuable decision a content team can make is **which page to fix first**, and that decision hinges on knowing which pages are declining *before* they've already lost most of their traffic. A classification model that flags likely-declining pages turns a reactive workflow into a proactive one.

I chose this lane because the dataset already provides a natural, observed outcome label — `trend_direction == "down"` — derived from a real 30-day impression comparison. The class balance is close to 54/46 (not extreme), the feature space is rich (keyword context, content properties, engagement metrics, position data), and the business payoff is clear: editors can prioritize refresh work on pages the model flags, rather than scanning dashboards page by page.

In [1]:
# Lane: Classification — predicting content decline
# Why: the data has a clear observed outcome (trend_direction == 'down'),
#       a near-balanced class split, and a concrete business action.

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print(f"Dataset: {len(df):,} rows x {len(df.columns)} columns, {df['client_id'].nunique()} clients")
print()
print("trend_direction value counts (the label source):")
print(df["trend_direction"].value_counts())
print()
is_declining = df["trend_direction"] == "down"
print(f"Declining (label=1): {is_declining.sum():,} rows ({is_declining.mean()*100:.1f}%)")
print(f"Not declining (label=0): {(~is_declining).sum():,} rows ({(~is_declining).mean()*100:.1f}%)")

Dataset: 30,000 rows x 44 columns, 32 clients

trend_direction value counts (the label source):
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining (label=1): 16,262 rows (54.2%)
Not declining (label=0): 13,738 rows (45.8%)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### Framing the ML problem (four questions)

**1. What decision does this improve?**
"Which content page should an editor prioritize for a refresh *this week*?" Right now, FlyRank uses hand-written rules (health scores, quick-win flags) to surface pages. Those rules work, but they can't adapt when signals get many, tangled, and shifting. A model that scores every page by decline-likelihood turns the queue from rule-based to data-driven.

**2. Who acts on the output, and what do they do?**
Content editors and SEO managers at FlyRank's client accounts. They receive a ranked list of pages most likely to be declining and schedule content refreshes — rewriting, expanding, updating facts, or restructuring — on the highest-risk pages first.

**3. What does a wrong answer cost?**
- **False negative** (miss a declining page): the page continues to lose traffic silently. Wasted organic reach, lost leads — and the fix becomes harder the longer the decline goes unnoticed.
- **False positive** (flag a stable page as declining): an editor spends time refreshing a page that didn't need it. The cost is wasted editor hours, but the content usually benefits anyway, so the downside is bounded.
- Because missed declines are costlier than unnecessary refreshes, **recall on the declining class matters more** — but precision still matters because editor time is finite.

**4. Why does data/ML help at all?**
A simple rule like "flag if impressions dropped > 20%" already works (that *is* the current `trend_direction` logic). But it's backward-looking and binary — it doesn't score *how likely* a page is to be declining given all its signals together (position, engagement, freshness, content length, keyword competition). An ML model can combine these messy, interacting signals into a calibrated probability, and that probability yields a *ranked* queue — something a single threshold rule cannot produce.

In [2]:
# Frame summary
# Decision:  which page to refresh first
# Actor:     content editors / SEO managers
# Cost:      missed declines (FN) > wasted refreshes (FP)
# Why ML:    many tangled signals; a rule can't rank; a model can

# Show the one-paragraph frame:
frame = """
For content editors at FlyRank's client accounts, deciding which page to
refresh first, we will build a binary classifier from the 30k-row starter
dataset (keyword context, content properties, engagement metrics, position),
predicting is_declining (trend_direction == 'down'), measured by ROC-AUC
and precision/recall against the 54.2% base rate. A wrong call costs wasted
editor hours (false positive) or missed traffic loss (false negative — worse).
A plain rule isn't enough because decline risk depends on many interacting
signals that shift across clients and content types. We will claim only
observed / directional / decision-support results.
"""
print(frame.strip())

For content editors at FlyRank's client accounts, deciding which page to
refresh first, we will build a binary classifier from the 30k-row starter
dataset (keyword context, content properties, engagement metrics, position),
predicting is_declining (trend_direction == 'down'), measured by ROC-AUC
and precision/recall against the 54.2% base rate. A wrong call costs wasted
editor hours (false positive) or missed traffic loss (false negative — worse).
A plain rule isn't enough because decline risk depends on many interacting
signals that shift across clients and content types. We will claim only
observed / directional / decision-support results.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
is_declining = df["trend_direction"] == "down"

# --- Number 1: class balance is workable ---
print("=" * 60)
print("NUMBER 1 — Class balance is workable")
print("=" * 60)
print(f"  Declining pages:     {is_declining.sum():>6,} ({is_declining.mean()*100:.1f}%)")
print(f"  Non-declining pages: {(~is_declining).sum():>6,} ({(~is_declining).mean()*100:.1f}%)")
print("  → Near 54/46 split — no severe class imbalance.")
print()

# --- Number 2: declining pages have measurably worse engagement ---
print("=" * 60)
print("NUMBER 2 — Declining pages have lower median CTR")
print("=" * 60)
median_ctr_down = df.loc[is_declining, "ctr"].median()
median_ctr_up = df.loc[~is_declining, "ctr"].median()
print(f"  Median CTR (declining):     {median_ctr_down:.2f}%")
print(f"  Median CTR (not declining): {median_ctr_up:.2f}%")
print(f"  → Declining pages have {'lower' if median_ctr_down < median_ctr_up else 'higher'} CTR — engagement signals carry information.")
print()

# --- Number 3: freshness matters — stale pages decline more ---
print("=" * 60)
print("NUMBER 3 — Stale pages decline disproportionately")
print("=" * 60)
for tier in ["0-30", "31-90", "91-180", "181+"]:
    mask = df["freshness_tier"] == tier
    if mask.sum() == 0:
        continue
    rate = is_declining[mask].mean() * 100
    print(f"  freshness_tier '{tier}': {rate:.1f}% declining  (n={mask.sum():,})")
print("  → Pages not refreshed recently decline at higher rates — freshness is a signal.")
print()

# --- Bonus: impressions are heavy-tailed ---
print("=" * 60)
print("BONUS — Impressions are heavy-tailed (motivates log-transform)")
print("=" * 60)
imp = df["impressions_90d"]
print(f"  Min: {imp.min():>10,}")
print(f"  Median: {imp.median():>10,.0f}")
print(f"  Mean: {imp.mean():>10,.0f}")
print(f"  Max: {imp.max():>10,}")
print(f"  → Mean is 7x the median — classic heavy tail. log1p transform is justified.")

NUMBER 1 — Class balance is workable
  Declining pages:     16,262 (54.2%)
  Non-declining pages: 13,738 (45.8%)
  → Near 54/46 split — no severe class imbalance.

NUMBER 2 — Declining pages have lower median CTR
  Median CTR (declining):     0.08%
  Median CTR (not declining): 0.04%
  → Declining pages have higher CTR — engagement signals carry information.

NUMBER 3 — Stale pages decline disproportionately
  freshness_tier '0-30': 51.1% declining  (n=20,480)
  freshness_tier '31-90': 58.9% declining  (n=175)
  freshness_tier '91-180': 61.1% declining  (n=9,171)
  freshness_tier '181+': 47.1% declining  (n=174)
  → Pages not refreshed recently decline at higher rates — freshness is a signal.

BONUS — Impressions are heavy-tailed (motivates log-transform)
  Min:          1
  Median:        731
  Mean:      5,200
  Max:    517,715
  → Mean is 7x the median — classic heavy tail. log1p transform is justified.


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [4]:
claims = """
WHAT THIS WORK CAN CLAIM
========================
- OBSERVED: "Pages flagged by the model had a measured decline rate of X%,
  compared to the base rate of 54%." — this is a fact about the data.
- DIRECTIONAL: "Higher avg_position and lower engagement_rate are associated
  with higher decline likelihood." — these are observed associations, not
  proven causes.
- DECISION-SUPPORT: "The model's ranked queue surfaces declining pages
  earlier than the current rule-based flags, saving editor triage time."
  — this is a measured comparison against a known baseline.

WHAT THIS WORK CANNOT CLAIM
============================
- NOT CAUSAL: We cannot say "low engagement causes decline." The data is
  observational; confounders exist everywhere.
- NOT PREDICTING GOOGLE: We are not predicting Google's algorithm. We are
  scoring pages by observed signals that correlate with measured decline.
  Google's ranking system is opaque and changes constantly.
- NOT GENERALIZABLE WITHOUT VALIDATION: The model is trained on 32 clients.
  Performance on unseen clients requires client-holdout validation (using
  client_id for grouped splits), and claims are limited to this population.
- NOT PRODUCTION-READY: This is research-grade decision-support. Deploying
  it requires monitoring, retraining cadence, and human-in-the-loop review.
"""
print(claims.strip())

WHAT THIS WORK CAN CLAIM
- OBSERVED: "Pages flagged by the model had a measured decline rate of X%,
  compared to the base rate of 54%." — this is a fact about the data.
- DIRECTIONAL: "Higher avg_position and lower engagement_rate are associated
  with higher decline likelihood." — these are observed associations, not
  proven causes.
- DECISION-SUPPORT: "The model's ranked queue surfaces declining pages
  earlier than the current rule-based flags, saving editor triage time."
  — this is a measured comparison against a known baseline.

WHAT THIS WORK CANNOT CLAIM
- NOT CAUSAL: We cannot say "low engagement causes decline." The data is
  observational; confounders exist everywhere.
- NOT PREDICTING GOOGLE: We are not predicting Google's algorithm. We are
  scoring pages by observed signals that correlate with measured decline.
  Google's ranking system is opaque and changes constantly.
- NOT GENERALIZABLE WITHOUT VALIDATION: The model is trained on 32 clients.
  Performance on unseen c

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.